In [1]:
import os
import torch
import torchaudio
import numpy as np
import pandas as pd

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [2]:
AUDIO_DIR  = "DATA/audio"
FRAME_DIR  = "DATA/images"   # ← single extracted frame per video


In [3]:
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [4]:
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=16000,
    n_fft=1024,
    hop_length=512,
    n_mels=64
)

db_transform = torchaudio.transforms.AmplitudeToDB()


In [5]:
emotion_map = {
    "01": 0, "02": 1, "03": 2, "04": 3,
    "05": 4, "06": 5, "07": 6, "08": 7
}

records = []

for fname in os.listdir(AUDIO_DIR):
    if not fname.endswith(".wav"):
        continue

    parts = fname.replace(".wav", "").split("-")
    if len(parts) != 7:
        continue

    img_path = os.path.join(FRAME_DIR, fname.replace(".wav", ".jpg"))
    if not os.path.exists(img_path):
        continue

    records.append({
        "audio": os.path.join(AUDIO_DIR, fname),
        "image": img_path,
        "emotion": emotion_map[parts[2]]
    })

df = pd.DataFrame(records)
print("Total samples:", len(df))


Total samples: 1440


In [6]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["emotion"],
    random_state=42
)


In [7]:
class MultimodalDataset(Dataset):
    def __init__(self, df, image_transform, training=True):
        self.df = df.reset_index(drop=True)
        self.image_transform = image_transform
        self.training = training

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # ---- IMAGE ----
        img = Image.open(row["image"]).convert("RGB")
        img = self.image_transform(img)

        # ---- AUDIO ----
        waveform, sr = torchaudio.load(row["audio"])


        waveform, sr = torchaudio.load(row["audio"])
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        
        if sr != 16000:
            waveform = torchaudio.functional.resample(waveform, sr, 16000)

        mel = mel_transform(waveform)
        mel = db_transform(mel)

        T = mel.shape[-1]
        if T < 128:
            mel = F.pad(mel, (0, 128 - T))
        else:
            mel = mel[:, :, :128]

        mel = (mel - mel.mean()) / (mel.std() + 1e-6)


        # 🔥 SpecAugment (TRAIN ONLY)
        if self.training and torch.rand(1) < 0.3:
            mel = torchaudio.transforms.FrequencyMasking(4)(mel)


        label = torch.tensor(row["emotion"], dtype=torch.long)
        return img, mel, label


In [8]:
train_ds = MultimodalDataset(train_df, image_transform, training=True)
val_ds   = MultimodalDataset(val_df, image_transform, training=False)

train_loader = DataLoader(
    train_ds, batch_size=16, shuffle=True, num_workers=0
)

val_loader = DataLoader(
    val_ds, batch_size=16, shuffle=False, num_workers=0
)


In [9]:
from sklearn.utils.class_weight import compute_class_weight

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(8),
    y=train_df["emotion"].values
)

weights = torch.tensor(weights, dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(
    weight=weights,
    label_smoothing=0.1
)


In [10]:
class ImageEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.resnet18(weights="IMAGENET1K_V1")
        self.features = nn.Sequential(*list(base.children())[:-1])
        self.fc = nn.Linear(512, 256)

    def forward(self, x):
        x = self.features(x).squeeze(-1).squeeze(-1)
        return self.fc(x)


In [11]:
class AudioEncoderTemporal(nn.Module):
    def __init__(self):
        super().__init__()

        # CNN frontend (local feature extraction)
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),   # [B, 32, 32, T/2]

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d((2, 2))    # [B, 64, 16, T/4]
        )

        # BiLSTM over time
        self.lstm = nn.LSTM(
            input_size=64 * 16,   # freq collapsed
            hidden_size=128,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(256, 256)  # 128*2 → 256

    def forward(self, x):
        # x: [B, 1, 64, T]
        x = self.cnn(x)               # [B, 64, 16, T']
        x = x.permute(0, 3, 1, 2)     # [B, T', 64, 16]
        x = x.reshape(x.size(0), x.size(1), -1)  # [B, T', 64*16]

        lstm_out, _ = self.lstm(x)    # [B, T', 256]
        x = lstm_out.mean(dim=1)      # temporal pooling

        return self.fc(x)


In [12]:
class FusionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 8)
        )

    def forward(self, img, aud):
        x = torch.cat([img, aud], dim=1)
        return self.fc(x)


In [13]:
image_encoder = ImageEncoder().to(device)
audio_encoder = AudioEncoderTemporal().to(device)
fusion_model  = FusionModel().to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.Adam(
    list(image_encoder.parameters()) +
    list(audio_encoder.parameters()) +
    list(fusion_model.parameters()),
    lr=3e-4
)


In [14]:
def train_epoch():
    image_encoder.train()
    audio_encoder.train()
    fusion_model.train()

    total, correct, loss_sum = 0, 0, 0

    for img, mel, y in train_loader:
        img, mel, y = img.to(device), mel.to(device), y.to(device)

        img_emb = image_encoder(img)
        aud_emb = audio_encoder(mel)

        # 🔥 Modality Dropout
        if torch.rand(1).item() < 0.15:
            img_emb = torch.zeros_like(img_emb)
        if torch.rand(1).item() < 0.15:
            aud_emb = torch.zeros_like(aud_emb)

        out = fusion_model(img_emb, aud_emb)
        loss = criterion(out, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_sum += loss.item()
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)

    return loss_sum / len(train_loader), correct / total


@torch.no_grad()
def validate():
    image_encoder.eval()
    audio_encoder.eval()
    fusion_model.eval()

    correct, total = 0, 0
    for img, mel, y in val_loader:
        img, mel, y = img.to(device), mel.to(device), y.to(device)
        out = fusion_model(
            image_encoder(img),
            audio_encoder(mel)
        )
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)

    return correct / total


In [15]:
EPOCHS = 15 
for epoch in range(EPOCHS): 
    loss, train_acc = train_epoch() 
    val_acc = validate() 
    print( 
        f"Epoch {epoch+1}/{EPOCHS} | " 
        f"Loss: {loss:.4f} | " 
        f"Train Acc: {train_acc:.3f} | " 
        f"Val Acc: {val_acc:.3f}" )

Epoch 1/15 | Loss: 1.8511 | Train Acc: 0.288 | Val Acc: 0.396
Epoch 2/15 | Loss: 1.4969 | Train Acc: 0.502 | Val Acc: 0.604
Epoch 3/15 | Loss: 1.3547 | Train Acc: 0.574 | Val Acc: 0.663
Epoch 4/15 | Loss: 1.1007 | Train Acc: 0.710 | Val Acc: 0.319
Epoch 5/15 | Loss: 1.0701 | Train Acc: 0.725 | Val Acc: 0.646
Epoch 6/15 | Loss: 0.9400 | Train Acc: 0.789 | Val Acc: 0.559
Epoch 7/15 | Loss: 0.8048 | Train Acc: 0.863 | Val Acc: 0.792
Epoch 8/15 | Loss: 0.9578 | Train Acc: 0.773 | Val Acc: 0.767
Epoch 9/15 | Loss: 0.8384 | Train Acc: 0.839 | Val Acc: 0.712
Epoch 10/15 | Loss: 0.8251 | Train Acc: 0.845 | Val Acc: 0.701
Epoch 11/15 | Loss: 0.7970 | Train Acc: 0.868 | Val Acc: 0.747
Epoch 12/15 | Loss: 0.8268 | Train Acc: 0.844 | Val Acc: 0.826
Epoch 13/15 | Loss: 0.7770 | Train Acc: 0.863 | Val Acc: 0.823
Epoch 14/15 | Loss: 0.6681 | Train Acc: 0.924 | Val Acc: 0.837
Epoch 15/15 | Loss: 0.7433 | Train Acc: 0.881 | Val Acc: 0.806


In [16]:
import torch
torch.save({
    "image": image_encoder.state_dict(),
    "audio": audio_encoder.state_dict(),
    "fusion": fusion_model.state_dict()
}, "model.pth")

print("✅ model.pth saved")


✅ model.pth saved
